<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting & Cleaning

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from shapely.geometry import shape
import requests
from google.colab import userdata
from google.colab import drive
from pprint import pprint as print

In [2]:
# Mount Google Drive
drive.mount('drive')

Mounted at drive


In [3]:
# Set up the API
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
}

TimeoutException: Requesting secret token timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Fetching a sample of data from the right time period
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

In [ ]:
# Collecting all the taxi data within the time period specified, if already collected use option 2
def load_raw_data(option = 1):
  if option == 1:
    query = """
    SELECT
      *
    WHERE
      tpep_pickup_datetime >= "2025-06-21T00:00:00"
      AND
      tpep_dropoff_datetime <= "2025-09-21T23:59:59"
    LIMIT 12000000
    """

    query_payload = {
        "query": query
    }

    query_response = requests.get(data_api, headers=headers, params=query_payload)
    print("All Done!!!")
    query_data = query_response.json()
    taxi_df = pd.DataFrame(query_data)
    return taxi_df
  elif option == 2:
    taxi_df = pd.read_csv("/content/drive/MyDrive/raw_taxi_summer_data.csv")
    return taxi_df
  else:
    print("Invalid option")
    return None

In [ ]:
# printing length of raw initial data
taxi_df:pd.DataFrame = load_raw_data(2)
print(len(taxi_df))

## Data Structure

In [ ]:
# Preview the first few rows using the dataframe head function
taxi_df.head()

In [ ]:
# Preview the last few rows using the dataframe head function
taxi_df.tail()

In [ ]:
# Displaying the Dataframes shape, columns, and amount of non null values using the info function
taxi_df.info()

In [ ]:
# Drop columns not needed for the project
taxi_df.drop(columns=['request_source', 'store_and_fwd_flag',':id',':version',':created_at',':updated_at'], axis=1, inplace=True)

In [ ]:
# Converting our two datetime columns to the pandas datetime type
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

## Missing and Duplicate Data

In [ ]:
# Remove duplicate records from the dataset
taxi_df = taxi_df.drop_duplicates().reset_index(drop=True)

In [ ]:
# collecting numeric columns to handle missing values
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [ ]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

In [ ]:
# Inspecting the ratecodeid for its values
print(taxi_df['ratecodeid'].value_counts(dropna=False))

In [ ]:
# Fill missing values in ratecodeid
taxi_df['ratecodeid'] = taxi_df['ratecodeid'].fillna(99)

In [ ]:
# Inspecting the congestion_surcharge for its values
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

In [ ]:
# Handiling congestion surchage values by replacing it with a boolean indicator
taxi_df['congestion_surcharge'].replace(
    {
      2.50: 1,
      0.00: 0,
      -2.50: 1,
      1.00: 1,
      0.75: 1
    },
    inplace=True
)

taxi_df['congestion_surcharge'] = taxi_df['congestion_surcharge'].fillna(-1)

In [ ]:
# Inspecting the airport_fee for its values
print(taxi_df['airport_fee'].value_counts(dropna=False))

In [ ]:
# Handiling airport_fee values by replacing it with a boolean indicator
taxi_df['airport_fee'].replace(
    {
      0.00: 0,
      1.75: 1,
      -1.75: 1,
      6.75: 1,
      5.00: 1,
      0.50: 1
    },
    inplace=True
)

taxi_df['airport_fee'] = taxi_df['airport_fee'].fillna(-1)

In [ ]:
# checking column values of when passenger count is empty
empty_passenger = taxi_df.loc[taxi_df['passenger_count'].isna()]
pmissing = (empty_passenger.isna().sum() / len(empty_passenger)) * 100
pmissing

In [ ]:
# checking length of data after removing duplicates and missing data
print(len(taxi_df))

## Outlier Treatment

In [ ]:
taxi_df.describe()

## Taxi Zones Data

In [ ]:
# Querying for taxi zones data
zones_query = {
    "query":"""
    SELECT
      *
    LIMIT 500
    """
}
zones_response = requests.get(
    "https://data.cityofnewyork.us/api/v3/views/8meu-9t5y/query.json",
    headers=headers,
    params=zones_query
  )
zone_data = zones_response.json()
print("All done!")

In [ ]:
zone_df = pd.DataFrame(zone_data)
zone_df["the_geom"] = zone_df["the_geom"].apply(lambda g: shape(g) if isinstance(g, dict) else None)
zone_df = geopandas.GeoDataFrame(zone_df, geometry="the_geom", crs="EPSG:4326")

In [ ]:
# Remove metadata columns
zone_df.drop(columns=[':id', ':version', ':created_at', ':updated_at'], axis=1, inplace=True)

In [ ]:
# fixing datatypes of three specific columns specified below
zone_df['shape_leng'] = zone_df['shape_leng'].astype("float")
zone_df['shape_area'] = zone_df['shape_area'].astype("float")
zone_df['locationid'] = zone_df['locationid'].astype("int")

In [ ]:
# checking our dataframes structure again
zone_df.info()

## Borough Validation

In [ ]:
# importing datasets from Google Drive
taxi_df = pd.read_csv("/content/drive/MyDrive/taxi_summer_data.csv")
zone_df = geopandas.read_file("/content/drive/MyDrive/taxi_zones_data.json")

In [ ]:
# merging out tables
taxi_zone_df = pd.merge(taxi_df, zone_df, left_on='pulocationid', right_on='locationid', how='left')
taxi_zone_df = pd.merge(taxi_zone_df, zone_df, left_on='dolocationid', right_on='locationid', how='left')
taxi_zone_df.rename(columns={'borough_x': 'pickup_borough', 'borough_y': 'dropoff_borough'}, inplace=True)
taxi_zone_df.drop(columns=['locationid_x', 'locationid_y'], inplace=True)
taxi_zone_df.info()

In [ ]:
# checking pickup borough values
taxi_zone_df['pickup_borough'].value_counts(dropna=False)

In [ ]:
# checking dropoff borough values
taxi_zone_df['dropoff_borough'].value_counts(dropna=False)

In [ ]:
# checking pickup borough values when NaN 
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

In [ ]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), 'pulocationid'].value_counts()


In [ ]:
# checking pickup borough values when NaN
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

In [ ]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), 'dolocationid'].value_counts()

In [ ]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['pickup_borough'] == 'EWR', 'pulocationid'].value_counts()


In [ ]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'] == 'EWR', 'dolocationid'].value_counts()

In [ ]:
# dropping taxi data where location id is specific values 265, 264, 57, 105, 1
taxi_df = taxi_df.loc[~taxi_df['pulocationid'].isin([265, 264, 57, 105, 1])]
taxi_df = taxi_df.loc[~taxi_df['dolocationid'].isin([265, 264, 57, 105, 1])]
taxi_df.info()


In [ ]:
# double checking ids
taxi_df.loc[taxi_df['pulocationid'].isin([265, 264, 57, 105, 1])]

In [ ]:
# double checking ids
taxi_df.loc[taxi_df['dolocationid'].isin([265, 264, 57, 105, 1])]

In [ ]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

## Export

In [ ]:
# Export the processed taxi records and spatial zone lookup tables to Google Drive
taxi_df.to_csv("/content/drive/MyDrive/taxi_summer_data.csv", index=False)
zone_df.to_file("/content/drive/MyDrive/taxi_zones_data.json", index=False)